# Quit-Risk Model From Presence History

This notebook builds an early-warning model from the daily presence workbook.

Important limitation:
- The workbook contains only **1 explicit `Sortie`** event.
- Because of that, we do **not** have enough clean quit labels to train a reliable classical attrition model.
- The target used below is a **proxy**: `target_quit_14d = 1` when an employee has an explicit exit in the next 14 days **or** disappears from the file within the next 14 days and then stays inactive for at least 14 more days.

So this notebook should be used as a **risk scoring / early warning** workflow, not as a final HR decision system.


In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_PATH = Path("presence_history_jan_to_8_avril_fixed_mise_en_demeure_logic.xlsx")
SHEET_NAME = "detail"
PREDICTION_HORIZON_DAYS = 14
INACTIVE_GAP_DAYS = 14
RANDOM_STATE = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Excel file not found: {DATA_PATH}")


In [5]:
raw_df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

standard_columns = [
    "employee_id",
    "employee_name",
    "group",
    "phone",
    "cc",
    "function_sap",
    "hire_date",
    "anciennete_raw",
    "supervisor",
    "segment",
    "gender",
    "nature",
    "entry_exit",
    "presence_hours",
    "motif",
    "eff_active",
    "eff_present",
    "eff_mr",
    "abs_p_per",
    "abs_np",
    "hours_sup",
    "late_count",
    "day",
    "month_name",
]

if len(raw_df.columns) != len(standard_columns):
    raise ValueError(f"Unexpected number of columns: {len(raw_df.columns)}")

rename_map = dict(zip(raw_df.columns, standard_columns))
df = raw_df.rename(columns=rename_map).copy()

month_map = {
    "Janvier": 1,
    "Fevrier": 2,
    "F?vrier": 2,
    "Mars": 3,
    "Avril": 4,
}

df["date"] = pd.to_datetime(
    {
        "year": 2026,
        "month": df["month_name"].map(month_map),
        "day": pd.to_numeric(df["day"], errors="coerce"),
    },
    errors="coerce",
)
df["hire_date"] = pd.to_datetime(df["hire_date"], errors="coerce")
df["anciennete_years"] = (
    df["anciennete_raw"].astype(str).str.replace(",", ".", regex=False)
)
df["anciennete_years"] = pd.to_numeric(df["anciennete_years"], errors="coerce")
df = df.sort_values(["employee_id", "date"]).reset_index(drop=True)

DATASET_END = df["date"].max()

print(f"Rows: {len(df):,}")
print(f"Employees: {df['employee_id'].nunique():,}")
print(f"Observation window: {df['date'].min().date()} -> {DATASET_END.date()}")
print("Entry/exit events:")
print(df["entry_exit"].value_counts(dropna=False))

display(df.head())


Rows: 1,976
Employees: 40
Observation window: 2026-01-01 -> 2026-04-08
Entry/exit events:
entry_exit
NaN       1972
Entrée       3
Sortie       1
Name: count, dtype: int64


,employee_id,employee_name,group,phone,cc,function_sap,hire_date,anciennete_raw,supervisor,segment,gender,nature,entry_exit,presence_hours,motif,eff_active,eff_present,eff_mr,abs_p_per,abs_np,hours_sup,late_count,day,month_name,date,anciennete_years
0,10349213,Naser Zaineb,G-320-2,58685018.0,20754974.0,Operator Assembly,2017-01-03,"9,0",Slimen Roukaya,Muster,Female,direct,NaN,8,NaN,1,1.0,NaN,NaN,NaN,0.0,NaN,1,Janvier,2026-01-01,9.0
1,10349213,Naser Zaineb,G-320-2,58685018.0,20754974.0,Operator Assembly,2017-01-03,"9,0",Slimen Roukaya,Muster,Female,direct,NaN,8,NaN,1,1.0,NaN,NaN,NaN,0.0,NaN,2,Janvier,2026-01-02,9.0
2,10349213,Naser Zaineb,G-320-2,58685018.0,20754974.0,Operator Assembly,2017-01-03,"9,0",Slimen Roukaya,Muster,Female,direct,NaN,0,Conge sans solde,1,NaN,NaN,1.0,NaN,NaN,NaN,3,Janvier,2026-01-03,9.0
3,10349213,Naser Zaineb,G-320-2,58685018.0,20754974.0,Operator Assembly,2017-01-03,"9,0",Slimen Roukaya,Muster,Female,direct,NaN,10,NaN,1,1.0,NaN,NaN,NaN,2.0,NaN,4,Janvier,2026-01-04,9.0
4,10349213,Naser Zaineb,G-320-2,58685018.0,20754974.0,Operator Assembly,2017-01-03,"9,0",Slimen Roukaya,Muster,Female,direct,NaN,10,NaN,1,1.0,NaN,NaN,NaN,2.0,NaN,5,Janvier,2026-01-05,9.0


In [6]:
employee_summary = (
    df.groupby("employee_id")
    .agg(
        employee_name=("employee_name", "first"),
        first_seen=("date", "min"),
        last_seen=("date", "max"),
        explicit_exit=("entry_exit", lambda s: (s == "Sortie").any()),
        explicit_entry=("entry_exit", lambda s: (s == "Entr?e").any()),
        rows=("employee_id", "size"),
    )
    .reset_index()
)
employee_summary["days_before_dataset_end"] = (DATASET_END - employee_summary["last_seen"]).dt.days

print("Employees with explicit Sortie:", int(employee_summary["explicit_exit"].sum()))
print("Employees not seen for 14+ days before dataset end:", int((employee_summary["days_before_dataset_end"] >= INACTIVE_GAP_DAYS).sum()))

display(employee_summary.sort_values(["explicit_exit", "days_before_dataset_end"], ascending=[False, False]).head(15))


Employees with explicit Sortie: 1
Employees not seen for 14+ days before dataset end: 28


,employee_id,employee_name,first_seen,last_seen,explicit_exit,explicit_entry,rows,days_before_dataset_end
5,10400023,Trabelsi Saber,2026-01-01,2026-03-18,True,False,77,21
33,10400593,HMIDI AZIZ,2026-01-01,2026-01-05,False,False,5,93
2,10377903,HAJEJ KHAIRI,2026-01-01,2026-01-06,False,False,6,92
0,10349213,Naser Zaineb,2026-01-01,2026-01-09,False,False,9,89
31,10400503,KHLIFI MOEZ,2026-01-01,2026-01-10,False,False,10,88
6,10400027,"Trabelsi, Rania",2026-01-01,2026-01-14,False,False,14,84
11,10400149,Groud Yassine,2026-01-01,2026-01-14,False,False,14,84
17,10400257,BACCAR ALI,2026-01-01,2026-01-14,False,False,14,84
18,10400286,MAKHLOUFI KENZA,2026-01-01,2026-01-14,False,False,14,84
26,10400389,Groud Marwen,2026-01-01,2026-01-14,False,False,14,84


## Snapshot Training Logic

Instead of training one row per employee, we create one row per **employee snapshot date**.
For each snapshot date we compute rolling features from the previous 14 days, then label whether that employee will:

1. have an explicit `Sortie` in the next 14 days, or
2. disappear from the file within the next 14 days and stay inactive long enough to confirm the disappearance.

That turns the problem into: **"Based on the recent attendance pattern, is this employee likely to leave soon?"**


In [7]:
def trailing_streak(values, predicate):
    streak = 0
    for value in reversed(values):
        if predicate(value):
            streak += 1
        else:
            break
    return streak


def build_feature_row(employee_history, snapshot_date):
    history = employee_history.loc[employee_history["date"] <= snapshot_date].tail(14).copy()

    motifs = history["motif"].fillna("")
    hours = pd.to_numeric(history["presence_hours"], errors="coerce").fillna(0)
    late = pd.to_numeric(history["late_count"], errors="coerce").fillna(0)
    overtime = pd.to_numeric(history["hours_sup"], errors="coerce").fillna(0)
    abs_np = pd.to_numeric(history["abs_np"], errors="coerce").fillna(0)
    abs_p = pd.to_numeric(history["abs_p_per"], errors="coerce").fillna(0)
    mr = pd.to_numeric(history["eff_mr"], errors="coerce").fillna(0)

    first_row = employee_history.iloc[0]

    return {
        "employee_id": first_row["employee_id"],
        "employee_name": first_row["employee_name"],
        "group": first_row["group"],
        "function_sap": first_row["function_sap"],
        "supervisor": first_row["supervisor"],
        "segment": first_row["segment"],
        "gender": first_row["gender"],
        "nature": first_row["nature"],
        "anciennete_years": first_row["anciennete_years"],
        "snapshot_date": snapshot_date,
        "days_since_start": int((snapshot_date - employee_history["date"].min()).days),
        "history_days": int(len(history)),
        "hours_sum_14d": float(hours.sum()),
        "hours_mean_14d": float(hours.mean()),
        "zero_hour_days_14d": int((hours == 0).sum()),
        "low_hour_days_14d": int((hours <= 4).sum()),
        "overtime_sum_14d": float(overtime.sum()),
        "delay_count_14d": float(late.sum()),
        "abs_np_sum_14d": float(abs_np.sum()),
        "abs_p_sum_14d": float(abs_p.sum()),
        "mr_sum_14d": float(mr.sum()),
        "mise_en_demeure_days_14d": int((motifs == "Mise en demeure").sum()),
        "sans_questionnaire_days_14d": int((motifs == "Sans questionnaire").sum()),
        "conge_sans_solde_days_14d": int((motifs == "Conge sans solde").sum()),
        "absence_autorise_days_14d": int((motifs == "Absence Autorise").sum()),
        "maladie_prolongee_days_14d": int((motifs == "Maladie prolongee").sum()),
        "recent_zero_streak": int(trailing_streak(hours.tolist(), lambda x: x == 0)),
        "recent_mise_en_demeure_streak": int(
            trailing_streak(motifs.tolist(), lambda x: x == "Mise en demeure")
        ),
        "current_month": int(snapshot_date.month),
    }


def build_training_snapshots(data, prediction_horizon_days=14, inactive_gap_days=14):
    rows = []

    for employee_id, employee_history in data.groupby("employee_id"):
        employee_history = employee_history.sort_values("date").copy()
        last_seen = employee_history["date"].max()
        explicit_exit_dates = set(
            employee_history.loc[employee_history["entry_exit"] == "Sortie", "date"]
        )

        for snapshot_date in employee_history["date"]:
            future_limit = snapshot_date + pd.Timedelta(days=prediction_horizon_days)

            # Drop snapshots that are too close to the dataset end because their future is censored.
            if future_limit > DATASET_END - pd.Timedelta(days=inactive_gap_days):
                continue

            explicit_exit_soon = any(snapshot_date < d <= future_limit for d in explicit_exit_dates)
            disappears_within_horizon = last_seen <= future_limit
            inactivity_is_confirmed = last_seen <= DATASET_END - pd.Timedelta(days=inactive_gap_days)

            row = build_feature_row(employee_history, snapshot_date)
            row["target_quit_14d"] = int(
                explicit_exit_soon or (disappears_within_horizon and inactivity_is_confirmed)
            )
            rows.append(row)

    return pd.DataFrame(rows)


def build_latest_snapshots(data):
    rows = []

    for employee_id, employee_history in data.groupby("employee_id"):
        employee_history = employee_history.sort_values("date").copy()
        rows.append(build_feature_row(employee_history, employee_history["date"].max()))

    latest = pd.DataFrame(rows)
    latest["days_since_last_seen"] = (DATASET_END - latest["snapshot_date"]).dt.days
    latest["is_recently_active"] = latest["days_since_last_seen"] <= 7
    return latest


In [8]:
snapshot_df = build_training_snapshots(
    df,
    prediction_horizon_days=PREDICTION_HORIZON_DAYS,
    inactive_gap_days=INACTIVE_GAP_DAYS,
)

print(f"Training snapshots: {len(snapshot_df):,}")
print(snapshot_df["target_quit_14d"].value_counts())
print(f"Positive rate: {snapshot_df['target_quit_14d'].mean():.2%}")

display(
    snapshot_df.loc[snapshot_df["target_quit_14d"] == 1, [
        "employee_id",
        "employee_name",
        "snapshot_date",
        "mise_en_demeure_days_14d",
        "recent_mise_en_demeure_streak",
        "abs_np_sum_14d",
        "zero_hour_days_14d",
    ]].head(10)
)


Training snapshots: 1,640
target_quit_14d
0    1271
1     369
Name: count, dtype: int64
Positive rate: 22.50%


,employee_id,employee_name,snapshot_date,mise_en_demeure_days_14d,recent_mise_en_demeure_streak,abs_np_sum_14d,zero_hour_days_14d
0,10349213,Naser Zaineb,2026-01-01,0,0,0.0,0
1,10349213,Naser Zaineb,2026-01-02,0,0,0.0,0
2,10349213,Naser Zaineb,2026-01-03,0,0,0.0,1
3,10349213,Naser Zaineb,2026-01-04,0,0,0.0,1
4,10349213,Naser Zaineb,2026-01-05,0,0,0.0,1
5,10349213,Naser Zaineb,2026-01-06,1,1,1.0,2
6,10349213,Naser Zaineb,2026-01-07,2,2,2.0,3
7,10349213,Naser Zaineb,2026-01-08,3,3,3.0,4
8,10349213,Naser Zaineb,2026-01-09,4,4,4.0,5
15,10367289,Glili Yassine,2026-01-07,0,0,1.0,1


In [9]:
feature_columns = [
    c
    for c in snapshot_df.columns
    if c not in ["employee_id", "employee_name", "snapshot_date", "target_quit_14d"]
]
X = snapshot_df[feature_columns].copy()
y = snapshot_df["target_quit_14d"].copy()
groups = snapshot_df["employee_id"].copy()

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.50).astype(int)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Test positives: {int(y_test.sum())}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"Average precision: {average_precision_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_stay", "actual_quit_proxy"],
    columns=["pred_stay", "pred_quit_proxy"],
)
display(cm)


Train rows: 1,158
Test rows: 482
Test positives: 73
ROC AUC: 0.854
Average precision: 0.598

              precision    recall  f1-score   support

           0      0.952     0.675     0.790       409
           1      0.307     0.808     0.445        73

    accuracy                          0.695       482
   macro avg      0.630     0.742     0.617       482
weighted avg      0.854     0.695     0.738       482



,pred_stay,pred_quit_proxy
actual_stay,276,133
actual_quit_proxy,14,59


In [10]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients),
})

print("Top features pushing risk UP:")
display(coef_df.sort_values("coefficient", ascending=False).head(12))

print("Top features pushing risk DOWN:")
display(coef_df.sort_values("coefficient", ascending=True).head(12))


Top features pushing risk UP:


,feature,coefficient,abs_coefficient
9,num__abs_np_sum_14d,2.278459,2.278459
44,cat__supervisor_renfort BMW,1.594905,1.594905
22,cat__group_G-300-1 NEBEN,1.355587,1.355587
32,cat__function_sap_Operator Packing,1.355587,1.355587
27,cat__group_G-320-1,1.277935,1.277935
30,cat__function_sap_Operator Assembly,1.088646,1.088646
47,cat__segment_Muster,1.077136,1.077136
28,cat__group_G-320-2,1.077136,1.077136
43,cat__supervisor_Slimen Roukaya,1.077136,1.077136
18,num__recent_mise_en_demeure_streak,0.912162,0.912162


Top features pushing risk DOWN:


,feature,coefficient,abs_coefficient
24,cat__group_G-300-3 NEBEN,-1.539403,1.539403
31,cat__function_sap_Operator Electrical Test,-1.364550,1.364550
26,cat__group_G-301-1,-1.354086,1.354086
38,cat__supervisor_Ben Fkhi Ali Saifeddine,-1.326191,1.326191
45,cat__segment_Cleaness,-1.326191,1.326191
14,num__conge_sans_solde_days_14d,-1.095168,1.095168
8,num__delay_count_14d,-0.914397,0.914397
40,cat__supervisor_HARZALLAH ARAFET G1,-0.819434,0.819434
49,cat__gender_Female,-0.807954,0.807954
16,num__maladie_prolongee_days_14d,-0.773034,0.773034


In [11]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]
)
final_model.fit(X, y)

latest_df = build_latest_snapshots(df)
latest_df["quit_risk_probability"] = final_model.predict_proba(latest_df[feature_columns])[:, 1]

recent_risk_table = latest_df.loc[latest_df["is_recently_active"]].sort_values(
    "quit_risk_probability", ascending=False
)

print("Latest risk view for recently active employees (last seen within 7 days):")
display(
    recent_risk_table[[
        "employee_id",
        "employee_name",
        "snapshot_date",
        "days_since_last_seen",
        "quit_risk_probability",
        "recent_mise_en_demeure_streak",
        "mise_en_demeure_days_14d",
        "abs_np_sum_14d",
        "zero_hour_days_14d",
        "hours_mean_14d",
    ]]
)


Latest risk view for recently active employees (last seen within 7 days):


,employee_id,employee_name,snapshot_date,days_since_last_seen,quit_risk_probability,recent_mise_en_demeure_streak,mise_en_demeure_days_14d,abs_np_sum_14d,zero_hour_days_14d,hours_mean_14d
12,10400198,CHERNI NIZAR,2026-04-03,5,1.000000,4,4,6.0,6,4.714286
7,10400066,Chebbi Mohamed,2026-04-08,0,0.999175,1,1,2.0,3,6.571429
30,10400488,Mghirbi Rami,2026-04-08,0,0.824258,0,0,0.0,1,7.142857
37,10400627,Rekik Kais,2026-04-08,0,0.287155,0,0,1.0,3,6.142857
39,10400657,BEN HMIDA HEDI,2026-04-08,0,0.248946,0,0,0.0,2,7.000000
23,10400349,Chebbi Mouna,2026-04-08,0,0.239171,0,0,0.0,2,7.000000
8,10400085,Gdoura Lotfi,2026-04-08,0,0.173095,0,0,0.0,2,6.428571
15,10400243,Sassi Imed,2026-04-08,0,0.165744,0,0,1.0,1,7.857143
29,10400409,Groud Hedi,2026-04-08,0,0.143147,0,0,0.0,2,6.857143
19,10400314,Harfati Ines,2026-04-08,0,0.048720,0,0,0.0,1,7.714286
